# MIT 805 Big Data Semester Project — Part 2
## Demand and Trip Economics Analysis

### Processing Objective

This analysis investigates how demand patterns relate to trip economics in New York City's
High Volume For-Hire Vehicle (HVFHV) market during 2025.

### Central Project Question

**How do demand patterns relate to trip economics and service performance in New York
City's HVFHV market during 2025?**

### Demand and Trip Economics Question

**How do passenger fares and driver compensation vary across different demand patterns
in the HVFHV market during 2025?**

The analysis will use PySpark to process the large trip-level dataset. Demand will first
be measured using trip volumes across time. The analysis will then investigate how
passenger fares, driver pay, trip distance and trip duration vary across demand conditions.

The analysis will demonstrate MapReduce-style distributed processing through:

1. **Mapping/Transformation** — creating time variables and trip-economic measures.
2. **Grouping/Shuffle** — grouping trips by meaningful time and operator keys.
3. **Reduction/Aggregation** — calculating trip counts, average fares, driver pay,
   distance and duration.
4. **Distributed Execution** — examining Spark execution plans, stages and shuffle
   operations.
5. **Visualization and Interpretation** — using reduced Spark outputs to communicate
   demand and economic patterns.

The main processing will remain in PySpark. Pandas will only be used where appropriate
for small aggregated results and final visualizations.

In [2]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
C:\Users\Lucky


In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, types as T

DATA_DIR = "data/raw"
OUTPUT_DIR = "output/part2"
FIGURE_DIR = "figures/part2"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print("Working directory:", os.getcwd())
print("Data directory:", os.path.abspath(DATA_DIR))
print("Part 2 setup ready")

Working directory: C:\Users\Lucky
Data directory: C:\Users\Lucky\data\raw
Part 2 setup ready


In [6]:
spark = (
    SparkSession.builder
    .appName("MIT805-Part2-Demand-Economics")
    .master("local[4]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Cores:", spark.sparkContext.defaultParallelism)

Spark version: 3.5.1
Cores: 4


In [7]:
months = range(1, 13)

paths = [
    os.path.join(
        DATA_DIR,
        f"fhvhv_tripdata_2025-{m:02d}.parquet"
    )
    for m in months
]

for path in paths:
    status = "EXISTS" if os.path.exists(path) else "MISSING"
    print(os.path.basename(path), status)

print(
    "\nFiles found:",
    sum(os.path.exists(p) for p in paths),
    "/",
    len(paths)
)

fhvhv_tripdata_2025-01.parquet EXISTS
fhvhv_tripdata_2025-02.parquet EXISTS
fhvhv_tripdata_2025-03.parquet EXISTS
fhvhv_tripdata_2025-04.parquet EXISTS
fhvhv_tripdata_2025-05.parquet EXISTS
fhvhv_tripdata_2025-06.parquet EXISTS
fhvhv_tripdata_2025-07.parquet EXISTS
fhvhv_tripdata_2025-08.parquet EXISTS
fhvhv_tripdata_2025-09.parquet EXISTS
fhvhv_tripdata_2025-10.parquet EXISTS
fhvhv_tripdata_2025-11.parquet EXISTS
fhvhv_tripdata_2025-12.parquet EXISTS

Files found: 12 / 12


In [8]:
# Load the 2025 HVFHV processing dataset
df_raw = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(*paths)
)

# Select columns required for demand and trip economics
df_p2 = df_raw.select(
    "hvfhs_license_num",
    "request_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "driver_pay",
    "congestion_surcharge",
    "cbd_congestion_fee"
)

print("Columns selected:", len(df_p2.columns))
df_p2.printSchema()

Columns selected: 12
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## 2. Map / Transformation Stage

The first stage transforms each individual trip record into variables required for
the analysis. Time features are extracted from the pickup timestamp to identify demand
patterns by month, day and hour.

Economic measures are also derived from valid trip records. These include passenger
fare per mile, driver pay per mile, driver pay per minute, and the proportion of the
base passenger fare represented by driver pay.

These transformations are performed in PySpark before any grouping or aggregation.

In [9]:
df_mapped = (
    df_p2

    # Time features
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("day_of_week", F.date_format("pickup_datetime", "EEEE"))
    .withColumn("hour", F.hour("pickup_datetime"))

    # Convert trip time from seconds to minutes
    .withColumn(
        "trip_minutes",
        F.col("trip_time") / 60.0
    )

    # Passenger fare per mile
    .withColumn(
        "fare_per_mile",
        F.when(
            (F.col("trip_miles") > 0) &
            (F.col("base_passenger_fare") >= 0),
            F.col("base_passenger_fare") / F.col("trip_miles")
        )
    )

    # Driver pay per mile
    .withColumn(
        "driver_pay_per_mile",
        F.when(
            (F.col("trip_miles") > 0) &
            (F.col("driver_pay") > 0),
            F.col("driver_pay") / F.col("trip_miles")
        )
    )

    # Driver pay per minute
    .withColumn(
        "driver_pay_per_minute",
        F.when(
            (F.col("trip_time") > 0) &
            (F.col("driver_pay") > 0),
            F.col("driver_pay") / (F.col("trip_time") / 60.0)
        )
    )

    # Driver pay relative to base passenger fare
    .withColumn(
        "driver_pay_fare_ratio",
        F.when(
            (F.col("base_passenger_fare") > 0) &
            (F.col("driver_pay") >= 0),
            F.col("driver_pay") / F.col("base_passenger_fare")
        )
    )
)

print("Mapped columns:", len(df_mapped.columns))

df_mapped.select(
    "pickup_datetime",
    "pickup_date",
    "month",
    "day_of_week",
    "hour",
    "trip_miles",
    "trip_minutes",
    "base_passenger_fare",
    "driver_pay",
    "fare_per_mile",
    "driver_pay_per_mile",
    "driver_pay_per_minute",
    "driver_pay_fare_ratio"
).printSchema()

Mapped columns: 21
root
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_minutes: double (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- fare_per_mile: double (nullable = true)
 |-- driver_pay_per_mile: double (nullable = true)
 |-- driver_pay_per_minute: double (nullable = true)
 |-- driver_pay_fare_ratio: double (nullable = true)



## 3. Shuffle and Reduce Stage — Hourly Demand

Demand is measured as the number of HVFHV trips picked up during each hour of each
calendar date in 2025.

Spark groups the transformed trip records by pickup date and hour. Records with the
same date-hour key are brought together through the shuffle process, after which the
number of trips is aggregated for each group.

This converts the trip-level dataset into a much smaller hourly demand dataset that can
be used to study demand patterns and classify demand conditions.

In [10]:
hourly_demand = (
    df_mapped
    .groupBy("pickup_date", "hour")
    .agg(
        F.count("*").alias("trip_count")
    )
    .orderBy("pickup_date", "hour")
)

print("Hourly demand DataFrame created.")

Hourly demand DataFrame created.


In [11]:
n_hourly = hourly_demand.count()

print(f"Number of date-hour observations: {n_hourly:,}")

hourly_demand.show(20, truncate=False)

Number of date-hour observations: 8,760
+-----------+----+----------+
|pickup_date|hour|trip_count|
+-----------+----+----------+
|2025-01-01 |0   |56860     |
|2025-01-01 |1   |61048     |
|2025-01-01 |2   |57377     |
|2025-01-01 |3   |47302     |
|2025-01-01 |4   |32819     |
|2025-01-01 |5   |21773     |
|2025-01-01 |6   |17501     |
|2025-01-01 |7   |16159     |
|2025-01-01 |8   |14101     |
|2025-01-01 |9   |13876     |
|2025-01-01 |10  |15635     |
|2025-01-01 |11  |17950     |
|2025-01-01 |12  |19866     |
|2025-01-01 |13  |22375     |
|2025-01-01 |14  |25209     |
|2025-01-01 |15  |26897     |
|2025-01-01 |16  |25941     |
|2025-01-01 |17  |27611     |
|2025-01-01 |18  |27556     |
|2025-01-01 |19  |26278     |
+-----------+----+----------+
only showing top 20 rows

